# 01_Ingesta_Bronze

Ingesta de los cuatro archivos CSV del proyecto desde el Unity Catalog Volume hacia tablas Delta en la capa Bronze.

El objetivo de esta capa es conservar los datos de negocio sin transformaciones semánticas. Únicamente se agregan metadatos de trazabilidad y se normalizan nombres de columnas para garantizar compatibilidad con Delta Lake.


## 1. Configuración

Se definen el catálogo, el schema Bronze y la ruta del Volume donde se encuentran los archivos raw.


In [ ]:
from pathlib import Path
import sys

# Ejecutar desde el Git Folder completo, no desde un notebook importado aislado.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "accidentalidad" / "policies.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontró el paquete accidentalidad; abra el repositorio completo en un Git Folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from accidentalidad.policies import sql_identifier, parse_bool, nonnegative_int

import uuid
from pyspark.sql import functions as F
from accidentalidad.quality import QualityGate

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catálogo")
dbutils.widgets.dropdown("enforce_reference_counts", "true", ["true", "false"], "Conteos del lote de referencia")
PROY_CATALOG = sql_identifier(dbutils.widgets.get("catalog_name"))
PROY_SCHEMA_BRONZE = "bronze"
RAW_PATH = f"/Volumes/{PROY_CATALOG}/bronze/raw_files"
ENFORCE_COUNTS = parse_bool(dbutils.widgets.get("enforce_reference_counts"))
RUN_ID = uuid.uuid4().hex
EXPECTED_COUNTS = {"accidentes": 23212, "causas": 23212, "vehiculos": 46128, "victimas": 11328}

## 2. Validación de archivos raw

Antes de ingerir los datos, se verifica que los cuatro archivos fuente estén disponibles en el Volume.


In [ ]:
archivos = {entry.name.rstrip("/") for entry in dbutils.fs.ls(RAW_PATH)}
faltantes = {f"{name}.csv" for name in EXPECTED_COUNTS} - archivos
if faltantes:
    raise FileNotFoundError(f"Faltan archivos en {RAW_PATH}: {sorted(faltantes)}")

## 3. Lectura con contrato de esquema
Los cuatro archivos se leen en modo `FAILFAST`, sin inferir tipos. Las columnas requeridas
se verifican antes de escribir. El lote raw debe ser una instantánea completa e inmutable
durante toda la ejecución; no se admite entregar únicamente los nuevos registros.

In [ ]:
def cargar_bronze(nombre_archivo, nombre_tabla, renombres):
    df = (
        spark.read.option("header", True).option("inferSchema", False)
        .option("mode", "FAILFAST").option("encoding", "UTF-8")
        .csv(f"{RAW_PATH}/{nombre_archivo}")
        .select("*", F.col("_metadata.file_path").alias("_source_file"))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    )
    missing = set(renombres) - set(df.columns)
    if missing:
        raise ValueError(f"{nombre_tabla}: columnas de origen ausentes: {sorted(missing)}")
    for original, nuevo in renombres.items():
        df = df.withColumnRenamed(original, nuevo)
    if len({name.lower() for name in df.columns}) != len(df.columns):
        raise ValueError(f"{nombre_tabla}: nombres de columnas duplicados")
    return df

## 4. Normalización técnica de columnas

En Bronze no se modifican los valores de negocio ni se tipan fechas, coordenadas o métricas. Solo se normalizan nombres de columnas para hacerlos compatibles y consistentes en Delta Lake.

La columna `_c0` se conserva intencionalmente en Bronze y se eliminará en Silver.


In [ ]:
renombres_accidentes = {
    "Croquis": "croquis",
    "Fecha accidente": "fecha_accidente",
    "Año accidente": "anio_accidente",
    "Mes accidente": "mes_accidente",
    "Dia accidente": "dia_accidente",
    "hora accidente": "hora_accidente",
    "Gravedad": "gravedad",
    "Clase accidente": "clase_accidente",
    "Localidad": "localidad",
    "Sitio exacto": "sitio_exacto",
    "Nro heridos": "nro_heridos",
    "Numero muertos": "numero_muertos",
    "Placa agente": "placa_agente",
    "Apellidos": "apellidos",
    "Nombres": "nombres",
    "Coordenada X": "coordenada_x",
    "Coordenada Y": "coordenada_y"
}

renombres_causas = {
    "Croquis": "croquis",
    "Fecha accidente": "fecha_accidente",
    "hipotesis/causa": "hipotesis_causa"
}

renombres_vehiculos = {
    "Corquis": "croquis",
    "Fecha accidente": "fecha_accidente",
    "Año accidente": "anio_accidente",
    "Mes accidente": "mes_accidente",
    "Dia accidente": "dia_accidente",
    "Hora accidente": "hora_accidente",
    "Direccion": "direccion",
    "Placa": "placa",
    "Clase": "clase",
    "Servicio": "servicio",
    "Clase accidente": "clase_accidente",
    "Corrdenada X": "coordenada_x",
    "Coordenada Y": "coordenada_y",
    "Fecha radica": "fecha_radica"
}

renombres_victimas = {
    "Corquis": "croquis",
    "Fecha accidente": "fecha_accidente",
    "Año accidente": "anio_accidente",
    "Mes accidente": "mes_accidente",
    "Dia accidente": "dia_accidente",
    "Hora accidente": "hora_accidente",
    "Clase": "clase",
    "Direccion": "direccion",
    "Descripcion victima": "descripcion_victima",
    "Gravedad": "gravedad",
    "Id usuario": "id_usuario",
    "Sexo": "sexo",
    "Fecha nacimiento": "fecha_nacimiento",
    "Placa": "placa",
    "Clase.1": "clase_1",
    "Servicio": "servicio",
    "Edad": "edad",
    "Coordenada X": "coordenada_x",
    "Coordenada Y": "coordenada_y"
}


## 5. Ingesta de los cuatro datasets


In [ ]:
df_accidentes_bronze = cargar_bronze(
    "accidentes.csv",
    "accidentes",
    renombres_accidentes
)

df_causas_bronze = cargar_bronze(
    "causas.csv",
    "causas",
    renombres_causas
)

df_vehiculos_bronze = cargar_bronze(
    "vehiculos.csv",
    "vehiculos",
    renombres_vehiculos
)

df_victimas_bronze = cargar_bronze(
    "victimas.csv",
    "victimas",
    renombres_victimas
)


## 6. Validación y publicación
El lote de referencia exige los conteos documentados. Para otra instantánea completa,
use `enforce_reference_counts=false`; se siguen exigiendo archivos no vacíos y claves presentes.
La publicación de las cuatro tablas ocurre solo si todos los controles pasan.

In [ ]:
tablas_bronze = {
    "accidentes": df_accidentes_bronze, "causas": df_causas_bronze,
    "vehiculos": df_vehiculos_bronze, "victimas": df_victimas_bronze,
}
gate = QualityGate(spark, PROY_CATALOG, RUN_ID, "bronze")
for nombre, df in tablas_bronze.items():
    total = df.count()
    gate.check(nombre, "empty_dataset", total == 0, False)
    if ENFORCE_COUNTS:
        gate.check(nombre, "reference_count", total, EXPECTED_COUNTS[nombre])
    invalid_keys = df.filter(F.col("croquis").isNull() | (F.trim("croquis") == "")).count()
    gate.check(nombre, "null_or_empty_croquis", invalid_keys)
gate.finish()  # Persiste la auditoría y lanza error antes de publicar si hay incumplimientos.

for nombre, df in tablas_bronze.items():
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{PROY_CATALOG}.bronze.{nombre}"))
    print(f"[OK] Publicado Bronze: {nombre}, run_id={RUN_ID}")

Conteos esperados:

- `accidentes`: 23,212
- `causas`: 23,212
- `vehiculos`: 46,128
- `victimas`: 11,328
